# Prueba del Sistema de Respuestas de Email

Este notebook te permite probar el sistema de recepción y procesamiento de respuestas de correo electrónico.

## 1. Importar dependencias

In [1]:
import os
import httpx
import asyncio
from dotenv import load_dotenv

load_dotenv(override=True)

True

## 2. Simular una respuesta de correo entrante

In [ ]:
# Datos de prueba que simulan una respuesta de Resend
test_email_response = {
    "type": "email",
    "id": "test_email_001",
    "from": "cliente.interesado@empresa.com",
    "to": "leads@pinbraae.resend.app",  # Tu dirección de Resend
    "subject": "Re: Email de ventas - ComplAI",
    "text": """Hola,

Gracias por contactarnos. Estoy muy interesado en su herramienta de cumplimiento SOC2.

Tenemos algunas preguntas:
1. ¿Cuál es el precio para una empresa de 50 empleados?
2. ¿Cuánto tiempo toma la implementación?
3. ¿Ofrecen soporte técnico 24/7?

Me gustaría agendar una demostración la próxima semana.

Saludos cordiales,
Juan Pérez
CTO, Empresa Tech""",
    "timestamp": "2024-01-01T15:30:00.000Z"
}

## 3. Enviar la respuesta simulada al webhook

In [ ]:
async def send_test_webhook():
    """Envía un email de prueba al webhook local"""
    
    webhook_url = "http://localhost:8000/webhook"
    
    async with httpx.AsyncClient() as client:
        try:
            response = await client.post(
                webhook_url,
                json=test_email_response,
                timeout=30.0
            )
            
            print(f"Estado: {response.status_code}")
            print(f"Respuesta: {response.json()}")
            
        except httpx.ConnectError:
            print("❌ Error: No se pudo conectar al webhook.")
            print("Asegúrate de que email_response_handler.py esté corriendo:")
            print("   uv run email_response_handler.py")
        except Exception as e:
            print(f"❌ Error: {str(e)}")

# Ejecutar
await send_test_webhook()

Estado: 200
Respuesta: {'status': 'success', 'message': 'Email processed'}


## 4. Ver los correos recibidos

In [4]:
async def get_received_emails():
    """Obtiene todos los correos recibidos"""
    
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get("http://localhost:8000/emails")
            emails = response.json()
            
            print(f"📧 Correos recibidos: {len(emails['emails'])}")
            print("=" * 60)
            
            for i, email in enumerate(emails['emails'], 1):
                print(f"\n{i}. De: {email['from']}")
                print(f"   Asunto: {email['subject']}")
                print(f"   Procesado: {'✅ Sí' if email.get('processed') else '❌ No'}")
                
                if email.get('analysis'):
                    print(f"\n   🤖 Análisis del Agente:")
                    print(f"   {email['analysis'][:200]}...")
                
                print("-" * 60)
                
        except Exception as e:
            print(f"❌ Error: {str(e)}")

# Ejecutar
await get_received_emails()

📧 Correos recibidos: 2

1. De: cliente@empresa.com
   Asunto: Re: Propuesta de Servicios
   Procesado: ✅ Sí

   🤖 Análisis del Agente:
   1. **Sentimiento:** Positivo  
2. **Nivel de interés:** Alto  
3. **Objeciones o preguntas principales:** Pregunta sobre precios y solicitud de una llamada  
4. **Sugerencia de respuesta:**  
   "Hola...
------------------------------------------------------------

2. De: cliente.interesado@empresa.com
   Asunto: Re: Email de ventas - ComplAI
   Procesado: ✅ Sí

   🤖 Análisis del Agente:
   1. **Sentimiento:** Positivo  
   
2. **Nivel de interés:** Alto  

3. **Objeciones o preguntas principales:**
   - Precio para una empresa de 50 empleados.
   - Tiempo de implementación.
   - Disponi...
------------------------------------------------------------


## 5. Responder a un correo específico

In [5]:
async def reply_to_email(email_id: str, reply_text: str):
    """Responde a un correo recibido"""
    
    async with httpx.AsyncClient() as client:
        try:
            response = await client.post(
                f"http://localhost:8000/emails/{email_id}/reply",
                json={"reply_body": reply_text},
                timeout=30.0
            )
            
            result = response.json()
            
            if result.get('status') == 'success':
                print(f"✅ Respuesta enviada exitosamente")
                print(f"ID de respuesta: {result.get('reply_id')}")
            else:
                print(f"❌ Error: {result.get('message')}")
                
        except Exception as e:
            print(f"❌ Error: {str(e)}")

# Ejemplo de respuesta
reply_message = """Hola Juan,

¡Muchas gracias por tu interés en ComplAI!

Me encantaría responderte todas tus preguntas:

1. **Precio para 50 empleados**: $499/mes (con descuento por startup)
2. **Tiempo de implementación**: 2-3 semanas
3. **Soporte**: Sí, ofrecemos soporte 24/7 por chat y email

¿Te vendría bien una demostración el martes a las 10:00 AM o el jueves a las 3:00 PM?

Saludos,
Equipo ComplAI"""

# Responder al email de prueba
await reply_to_email("test_email_001", reply_message)

❌ Error: None


## 6. Probar con diferentes tipos de respuestas

In [6]:
# Diferentes escenarios de prueba

test_scenarios = [
    {
        "name": "Cliente interesado (Lead Caliente)",
        "email": {
            "type": "email",
            "id": "hot_lead_001",
            "from": "ceo@startup.com",
            "subject": "Re: ComplAI - Muy interesado",
            "text": "¡Esto es exactamente lo que necesitamos! ¿Podemos empezar mañana?",
            "timestamp": "2024-01-01T16:00:00.000Z"
        }
    },
    {
        "name": "Cliente con objeciones (Lead Tibio)",
        "email": {
            "type": "email",
            "id": "warm_lead_001",
            "from": "director@empresa.com",
            "subject": "Re: ComplAI",
            "text": "Suena interesante, pero el precio es muy alto. ¿Tienen algún plan más económico?",
            "timestamp": "2024-01-01T17:00:00.000Z"
        }
    },
    {
        "name": "No interesado (Lead Frío)",
        "email": {
            "type": "email",
            "id": "cold_lead_001",
            "from": "info@company.com",
            "subject": "Re: ComplAI",
            "text": "Gracias pero no estamos interesados en este momento.",
            "timestamp": "2024-01-01T18:00:00.000Z"
        }
    }
]

async def test_all_scenarios():
    """Prueba todos los escenarios"""
    
    async with httpx.AsyncClient() as client:
        for scenario in test_scenarios:
            print(f"\n📧 Probando: {scenario['name']}")
            print("-" * 60)
            
            try:
                response = await client.post(
                    "http://localhost:8000/webhook",
                    json=scenario['email'],
                    timeout=30.0
                )
                print(f"✅ Procesado: {response.json()}")
                await asyncio.sleep(1)  # Pequeña pausa entre tests
                
            except Exception as e:
                print(f"❌ Error: {str(e)}")

# Ejecutar todos los tests
await test_all_scenarios()


📧 Probando: Cliente interesado (Lead Caliente)
------------------------------------------------------------
✅ Procesado: {'status': 'success', 'message': 'Email processed'}

📧 Probando: Cliente con objeciones (Lead Tibio)
------------------------------------------------------------
✅ Procesado: {'status': 'success', 'message': 'Email processed'}

📧 Probando: No interesado (Lead Frío)
------------------------------------------------------------
✅ Procesado: {'status': 'success', 'message': 'Email processed'}


## 7. Ver todos los correos procesados

In [7]:
# Ejecutar de nuevo para ver todos los correos
await get_received_emails()

📧 Correos recibidos: 5

1. De: cliente@empresa.com
   Asunto: Re: Propuesta de Servicios
   Procesado: ✅ Sí

   🤖 Análisis del Agente:
   1. **Sentimiento:** Positivo  
2. **Nivel de interés:** Alto  
3. **Objeciones o preguntas principales:** Pregunta sobre precios y solicitud de una llamada  
4. **Sugerencia de respuesta:**  
   "Hola...
------------------------------------------------------------

2. De: cliente.interesado@empresa.com
   Asunto: Re: Email de ventas - ComplAI
   Procesado: ✅ Sí

   🤖 Análisis del Agente:
   1. **Sentimiento:** Positivo  
   
2. **Nivel de interés:** Alto  

3. **Objeciones o preguntas principales:**
   - Precio para una empresa de 50 empleados.
   - Tiempo de implementación.
   - Disponi...
------------------------------------------------------------

3. De: ceo@startup.com
   Asunto: Re: ComplAI - Muy interesado
   Procesado: ✅ Sí

   🤖 Análisis del Agente:
   1. **Sentimiento**: Positivo  
2. **Nivel de interés**: Alto  
3. **Objeciones o preguntas